# Reproduce all figures and headline numbers

**Tier 1 reproduction.** This notebook reads only from `../data/` and needs no network,
no API keys and no credentials. Run All should complete in a few seconds and regenerate
every figure in `../figures/` plus every number quoted in `../README.md`.

If any assertion below fails, the published data and the published claims have drifted
apart — please open an issue.

Requires: `pandas`, `matplotlib`.

In [ ]:
import pandas as pd, numpy as np, matplotlib as mpl, matplotlib.pyplot as plt
from pathlib import Path

DATA = Path("../data"); FIGDIR = Path("../figures"); FIGDIR.mkdir(exist_ok=True)

ORDER = ["Small molecule","Antibody","ADC","Peptide","Protein/Enzyme","Oligonucleotide","Other"]
COL = {"Small molecule":"#4C72B0","Antibody":"#C44E52","ADC":"#DD8452","Peptide":"#55A868",
       "Protein/Enzyme":"#8172B3","Oligonucleotide":"#64B5CD","Other":"#8C8C8C"}
SNAKE = {m: m.lower().replace("/","_").replace(" ","_") for m in ORDER}

mpl.rcParams.update({"figure.dpi":110, "font.size":9, "axes.titlesize":10,
                     "axes.spines.top":False, "axes.spines.right":False,
                     "axes.labelsize":9, "legend.frameon":False})

approvals   = pd.read_csv(DATA/"approvals_by_year.csv").set_index("year")
shares      = pd.read_csv(DATA/"approvals_share_by_year.csv").set_index("year")
by_era      = pd.read_csv(DATA/"approvals_by_era.csv").set_index("era")
substances  = pd.read_csv(DATA/"approvals_substance_level.csv")
withdrawals = pd.read_csv(DATA/"withdrawals_by_year.csv").set_index("withdrawal_year")
wd_detail   = pd.read_csv(DATA/"withdrawals_substance_level.csv")
wd_rate     = pd.read_csv(DATA/"withdrawal_rate_by_era.csv").set_index("era")

A = approvals[[SNAKE[m] for m in ORDER]]; A.columns = ORDER
W = withdrawals[[SNAKE[m] for m in ORDER]]; W.columns = ORDER
S = shares[[SNAKE[m] for m in ORDER]];     S.columns = ORDER
print(f"{len(substances)} substances, {approvals.index.min()}-{approvals.index.max()}; "
      f"{len(wd_detail)} withdrawals")

## 1. Headline numbers

Every claim in the README, recomputed from the tables.

In [ ]:
# --- internal consistency: aggregates must agree with the substance-level table ---
from_substances = (substances.groupby(["approval_year","final_class"]).size()
                   .unstack(fill_value=0).reindex(columns=ORDER, fill_value=0)
                   .reindex(A.index, fill_value=0))
assert (from_substances.values == A.values).all(), "by_year disagrees with substance-level table"
assert approvals["total"].equals(A.sum(1)), "total column disagrees with row sum"
assert np.allclose(S.sum(1), 100, atol=0.15), "shares do not sum to 100"

wd_from_detail = (wd_detail.groupby(["withdrawal_year","final_class"]).size()
                  .unstack(fill_value=0).reindex(columns=ORDER, fill_value=0)
                  .reindex(W.index, fill_value=0))
assert (wd_from_detail.values == W.values).all(), "withdrawals by year disagrees with detail"
print("aggregate/detail consistency: OK")

# --- headline claims ---
BIO = [m for m in ORDER if m not in ("Small molecule","Other")]
bio_share = (by_era[[SNAKE[m] for m in BIO]].sum(1) / by_era.sum(1) * 100).round(1)
print("\nBiologic share by era (%):"); print(bio_share.to_string())

sm_first = S.loc[1990:1999,"Small molecule"].mean(); sm_last = S.loc[2020:2025,"Small molecule"].mean()
print(f"\nSmall-molecule share: {sm_first:.1f}% (1990s) -> {sm_last:.1f}% (2020s)"
      f"  = {sm_first-sm_last:.0f} point fall")

ratio = A.sum().sum() / W.sum().sum()
print(f"\nApprovals {A.sum().sum()} : withdrawals {W.sum().sum()} = {ratio:.1f} : 1")
print("\nWithdrawal rate by era (%):"); print(wd_rate["withdrawal_rate_pct"].to_string())
assert wd_rate["withdrawal_rate_pct"].idxmax() == "2000-2009", "peak era claim is wrong"
print("\npeak withdrawal rate is in 2000-2009: confirmed")

In [ ]:
# --- per-modality withdrawal rate (the ADC caveat) ---
rate = pd.DataFrame({"approved": A.sum(), "withdrawn": W.sum()}).reindex(ORDER)
rate["rate_pct"] = (rate.withdrawn / rate.approved * 100).round(1)
print(rate.to_string())
print(f"\nADC rate {rate.loc['ADC','rate_pct']}% rests on "
      f"{rate.loc['ADC','withdrawn']} events / {rate.loc['ADC','approved']} approvals "
      f"- both later returned to market. Treat as a hypothesis, not a rate.")
print(f"returned to market: {int(wd_detail.returned.sum())} of {len(wd_detail)}")
print(f"formulation-scoped:  {int((wd_detail.scope=='formulation').sum())} of {len(wd_detail)}")

## 2. Figure 1 — approvals by modality

In [ ]:
fig = plt.figure(figsize=(13.4,8.6))
gs = fig.add_gridspec(2,2, height_ratios=[1.15,1], width_ratios=[2.25,1], hspace=0.42, wspace=0.22)
ax1 = fig.add_subplot(gs[0,:]); ax2 = fig.add_subplot(gs[1,0]); ax3 = fig.add_subplot(gs[1,1])

bot = np.zeros(len(A))
for m in ORDER:
    ax1.bar(A.index, A[m], bottom=bot, color=COL[m], width=0.76,
            edgecolor="white", linewidth=0.5, label=m)
    bot += A[m].values
for x, t in zip(A.index, A.sum(1)):
    ax1.text(x, t+0.8, str(int(t)), ha="center", va="bottom", fontsize=5.6, color="#444")
ax1.axvline(2005.5, color="#555", lw=0.9, ls=(0,(4,2)))
ax1.text(1997.6, 68, "1990-2005  lower confidence", fontsize=6.6, color="#666",
         ha="center", style="italic")
ax1.text(2015.6, 68, "2006-2025  validated", fontsize=6.6, color="#333",
         ha="center", style="italic")
ax1.set_ylabel("Novel active substances approved"); ax1.set_ylim(0,74)
ax1.set_xticks(A.index); ax1.set_xticklabels(A.index, rotation=90, fontsize=6)
ax1.set_title(f"Biologics rose from {bio_share.iloc[0]:.0f}% to {bio_share.iloc[-1]:.0f}%"
              " of FDA novel approvals across four eras", loc="left")
ax1.legend(ncol=7, loc="upper left", bbox_to_anchor=(0,-0.14), fontsize=7,
           handlelength=1.1, columnspacing=1.5)

for m in ORDER:
    ax2.plot(S.index, S[m], color=COL[m], lw=1.5 if m=="Small molecule" else 1.0,
             marker="o", ms=2.4)
# de-collide the end-of-line labels: several classes sit within a point of each other at 2025
lab = sorted(((S[m].iloc[-1], m) for m in ORDER), reverse=True)
MINGAP = 4.2
ypos = []
for v, m in lab:
    y = v if not ypos else min(v, ypos[-1] - MINGAP)
    ypos.append(y)
for (v, m), y in zip(lab, ypos):
    ax2.plot([S.index[-1], S.index[-1]+1.4], [v, y], color=COL[m], lw=0.5, alpha=0.6,
             clip_on=False)
    ax2.text(S.index[-1]+1.9, y, m, color=COL[m], fontsize=6.4, va="center")
ax2.axvline(2005.5, color="#555", lw=0.9, ls=(0,(4,2)))
ax2.set_ylabel("Share of that year's approvals (%)"); ax2.set_ylim(-3,103)
ax2.set_xlim(1989, 2033)
ax2.set_xticks(range(1990, 2026, 5))   # ticks stop at the last data year
ax2.set_title("Small-molecule share fell; antibodies went from near-absent to second-largest",
              loc="left", fontsize=8.6)

eshare = by_era[[SNAKE[m] for m in ORDER]].div(by_era.sum(1), axis=0)*100
eshare.columns = ORDER
yy = np.arange(len(ORDER))
for k, era in enumerate(eshare.index):
    ax3.barh(yy + (1.5-k)*0.2, eshare.loc[era, ORDER], height=0.19,
             color=[COL[m] for m in ORDER], alpha=0.30+0.233*k)
for i, m in enumerate(ORDER):
    ax3.text(eshare.iloc[-1][m]+1.2, yy[i]-0.30, f"{eshare.iloc[-1][m]:.0f}%",
             fontsize=6.4, va="center", fontweight="bold")
ax3.set_yticks(yy); ax3.set_yticklabels(ORDER, fontsize=7); ax3.invert_yaxis()
ax3.set_xlabel("Share of novel approvals in era (%)")
ax3.set_title("Era shares, palest = 1990s to solid = 2020s", loc="left", fontsize=8.6)

for ax, l in ((ax1,"a"),(ax2,"b"),(ax3,"c")):
    ax.text(-0.055, 1.06, l, transform=ax.transAxes, fontsize=12, fontweight="bold", va="top")
fig.savefig(FIGDIR/"approvals_by_modality_1990_2025.png", dpi=300, bbox_inches="tight")
print("saved approvals_by_modality_1990_2025.png")

## 3. Figure 2 — approvals and withdrawals, mirrored

Note the deliberate asymmetry in the caption: panel **a** is validated from source data, panel **b** is a curated list whose completeness cannot be benchmarked.

In [ ]:
fig2 = plt.figure(figsize=(13.8,8.6))
gs2 = fig2.add_gridspec(2,1, height_ratios=[3.05,1], hspace=0.06)
axA = fig2.add_subplot(gs2[0]); axB = fig2.add_subplot(gs2[1], sharex=axA)

bot = np.zeros(len(A))
for m in ORDER:
    axA.bar(A.index, A[m], bottom=bot, color=COL[m], width=0.76,
            edgecolor="white", linewidth=0.5, label=m)
    bot += A[m].values
for x,t in zip(A.index, A.sum(1)):
    axA.text(x, t+0.9, str(int(t)), ha="center", va="bottom", fontsize=5.5, color="#444")
axA.axvline(2005.5, color="#555", lw=0.9, ls=(0,(4,2)))
axA.set_ylabel("Novel substances APPROVED", labelpad=13); axA.set_ylim(0,76)
axA.tick_params(axis="x", length=0, labelbottom=False)
axA.spines["bottom"].set_visible(False)
axA.set_title("US drug approvals and withdrawals by modality, 1990-2025 - "
              f"withdrawals run {ratio:.0f}:1 behind approvals", loc="left")

MW = -W
bot = np.zeros(len(MW))
for m in ORDER:
    axB.bar(MW.index, MW[m], bottom=bot, color=COL[m], width=0.76,
            edgecolor="white", linewidth=0.5)
    bot += MW[m].values
for x,t in zip(W.index, W.sum(1)):
    if t: axB.text(x, -t-0.14, str(int(t)), ha="center", va="top", fontsize=5.8, color="#444")
for x in wd_detail.loc[wd_detail.returned,"withdrawal_year"].unique():
    axB.plot([x],[0.15], marker="^", ms=3.0, color="#333", clip_on=False, zorder=6)
axB.axvline(2005.5, color="#555", lw=0.9, ls=(0,(4,2)))
axB.axhline(0, color="#333", lw=0.9)
axB.set_ylabel("WITHDRAWN from market", labelpad=13)
axB.set_ylim(-5.0,0); axB.set_yticks([0,-1,-2,-3,-4])
axB.set_xticks(A.index); axB.set_xticklabels(A.index, rotation=90, fontsize=6)
axB.spines["top"].set_visible(False)
axB.text(2007.5, -4.62, "^ later returned to market   -   curated list, not exhaustive "
         "- completeness not benchmarked", fontsize=6.2, color="#8a4a4a",
         va="center", ha="center", style="italic")
for ax,l in ((axA,"a"),(axB,"b")):
    ax.text(-0.045, 1.04, l, transform=ax.transAxes, fontsize=12, fontweight="bold", va="top")
fig2.legend(handles=[mpl.patches.Patch(facecolor=COL[m], label=m) for m in ORDER],
            ncol=7, loc="lower center", bbox_to_anchor=(0.5,-0.04), fontsize=7.2,
            handlelength=1.2, columnspacing=1.6)
fig2.savefig(FIGDIR/"approvals_withdrawals_mirror_1990_2025.png",
             dpi=300, bbox_inches="tight")
print("saved approvals_withdrawals_mirror_1990_2025.png")

## 4. Re-running the shipped validation checks

`validation/validation_checks_approvals.csv` records the checks run during the original
analysis. They are reproduced here for inspection; the assertions in section 1 are the
live versions.

In [ ]:
checks = pd.read_csv("../validation/validation_checks_approvals.csv")
print(checks.to_string(index=False))
assert (checks["result"]=="PASS").all(), "a shipped check is not PASS"

recon = pd.read_csv("../validation/reconciliation_vs_fda_published.csv")
r = np.corrcoef(recon["my_drug_count"], recon["fda_published_novel"])[0,1]
print(f"\nreconciliation vs FDA published counts, 2006-2025: r = {r:.4f}")
print(f"exact matches: {int((recon['delta_drugs']==0).sum())}/{len(recon)}"
      f" | max |delta|: {int(recon['delta_drugs'].abs().max())}")
print("\nAll checks reproduced.")